In [ ]:
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

import warnings
warnings.simplefilter("ignore")

# Comparing Two Samples: A/B Testing

In [ ]:
births = Table.read_table('baby.csv')
smoking_and_birthweight = births.select('Maternal Smoker', 'Birth Weight')
smoking_and_birthweight.show(6)

In [ ]:
smoking_and_birthweight.hist('Birth Weight', group='Maternal Smoker')

In [ ]:
means_table = smoking_and_birthweight.group('Maternal Smoker', np.average)
means_table

In [ ]:
means_arr = means_table.column('Birth Weight average')
observed_diff = means_arr.item(0) - means_arr.item(1)
observed_diff

Null hypothesis: In the population, the distribution of birth weights of babies is the same for mothers who don’t smoke as for mothers who do. The difference in the sample is due to chance.

Alternative hypothesis: In the population, the babies of the mothers who smoke have a lower birth weight, on average, than the babies of the non-smokers.

# Review: Exploring Null Hypothesis with One Sample

Step 1. Write a function to simulate one occurrence of the test statistic.

Step 2. Collect many simulated instances of the test statistic (for loop and np.append).

Step 3. Load outcomes into a table (Table().with_column)

Step 4. Visualize the outcomes with a histogram (hist)

Step 5. Calculate the p-value of the observed statistic (np.count_nonzero).

# New: Exploring Null Hypothesis with Two Samples

Step 1. Write a function to simulate one occurrence of the test statistic WHICH COMPARES THE TWO SAMPLES.

Step 2. Collect many simulated instances of the test statistic BY PERMUTING CATEGORY LABELS OF THE DATA SET TO GENERATE VARIETY (sample without replacement to permute, for loop, and np.append).

Step 3. Load outcomes into a table (Table().with_column)

Step 4. Visualize the outcomes with a histogram (hist)

Step 5. Calculate the p-value of the observed statistic (np.count_nonzero).

# Step 1: Function for Test Statistic

### Shuffling labels example

In [ ]:
staff = Table().with_columns(
    'Names', make_array('Jim', 'Pam', 'Dwight', 'Michael'),
    'Ages', make_array(29, 28, 34, 41)
)
staff

In [ ]:
# Sampling without replacement means the same row cannot be selected twice.
# Sampling without providing a number of rows will produce a table with the same number of rows as the original.
# This example therefore produces a random permutation (shuffling) of the original table.
shuffled_staff = staff.sample(with_replacement = False)
shuffled_staff

In [ ]:
# You can now put the names from the original table onto the shuffled table, which results in shuffling just the ages.
original_names = staff.column('Names')
shuffled_staff = shuffled_staff.with_column('Names', original_names)
shuffled_staff

Write a function called difference_of_means. It permutes the True/False labels of the Smoking column, calculates the average of each group, then returns the difference of the two means.

In [ ]:
def difference_of_means():
    shuffled_tbl = smoking_and_birthweight.sample(with_replacement = False)
    original_smoke = smoking_and_birthweight.column('Maternal Smoker')
    shuffled_tbl = shuffled_tbl.with_column('Maternal Smoker', original_smoke)
    means_tbl = shuffled_tbl.group('Maternal Smoker', np.average)
    means_arr = means_tbl.column('Birth Weight average')
    diff = means_arr.item(0) - means_arr.item(1)
    return diff

In [ ]:
difference_of_means()

# Step 2: Iteration

In [ ]:
diffs_arr = make_array()

for _ in np.arange(1000):
    sim_diff = difference_of_means()
    diffs_arr = np.append(diffs_arr, sim_diff)

diffs_arr

# Step 3 and 4: Create a Table and Visualize the Data

In [ ]:
diffs_tbl = Table().with_column('Sim Diffs', diffs_arr)
diffs_tbl.hist(bins=20)

# Step 5: Calculate the p-value

Calculate the percent of all simulated differences that favor the alternative hypothesis.

In [ ]:
observed_diff

In [ ]:
num_alt = np.count_nonzero(diffs_arr >= observed_diff)
num_total = len(diffs_arr)
p_value = num_alt / num_total
p_value